In [14]:
import Pkg
Pkg.add(["MLDatasets", "Flux", "IJulia"])
Pkg.update(["MLDatasets", "Flux", "IJulia"])

ENV["DATADEPS_ALWAYS_ACCEPT"] = "true"

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
    Updating registry at `~/.julia/registries/General.toml`
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


"true"

In [ ]:
ENV["DATADEPS_ALWAYS_ACCEPT"] = "true" 

using MLDatasets
using Flux: onehotbatch
using Random

include("clothesolver.jl")

# 1. Dane i Preprocessing
train_data = MLDatasets.FashionMNIST(split=:train)
test_data  = MLDatasets.FashionMNIST(split=:test)

X_train = Float32.(reshape(train_data.features, 28, 28, 1, :))
X_test  = Float32.(reshape(test_data.features, 28, 28, 1, :))

Y_train_raw = train_data.targets
Y_test_raw  = test_data.targets

Y_train = Float32.(onehotbatch(Y_train_raw, 0:9))
Y_test  = Float32.(onehotbatch(Y_test_raw, 0:9))

println("Wymiary: ", size(X_train))

In [ ]:
my_net_def = Chain(
  Conv((3, 3), 1 => 6,  bias=false),
  MaxPool((2, 2)),
  Conv((3, 3), 6 => 16, bias=false),
  MaxPool((2, 2)),
  Flatten(),
  Dense(400 => 84, relu),
  Dropout(0.4),
  Dense(84 => 10)
)

# build_model now owns input/target/loss — no BATCH_SIZE constant needed.
my_model = build_model(my_net_def, (28, 28, 1); batch_size=32)
println("Model zbudowany, batch_size = $(my_model.batch_size)")

# --- Inference ---
function test(model::CompiledModel, X, Y_raw)
    correct     = 0
    N           = size(X, 4)
    bs          = model.batch_size        # concrete Int — type-stable
    input_node  = model.input
    final_layer = model.chain.layers[end]

    for i in 1:bs:N
        b_end    = min(i + bs - 1, N)
        actual_b = b_end - i + 1

        fill!(input_node.data, 0f0)
        copyto!(@view(input_node.data[:, :, :, 1:actual_b]), @view(X[:, :, :, i:b_end]))
        forward!(model.chain, input_node, false)

        for b in 1:actual_b
            pred_class = argmax(@view(final_layer.out.data[:, b])) - 1
            correct += (pred_class == Y_raw[i + b - 1])
        end
    end

    acc = round(correct / N * 100, digits=2)
    println("Skuteczność: ", acc, "%")
    return acc
end

# --- Training ---
# batch_size comes from model.batch_size (concrete Int field) — no untyped kwarg,
# no boxing on each of the 1875 loop iterations → ~2× faster than old ; batch_size form.
function train!(model::CompiledModel, X, Y; η::Float32)
    bs          = model.batch_size
    input_node  = model.input
    target_node = model.target
    loss_fn     = model.loss
    N           = size(X, 4)
    indices     = randperm(N)
    total_loss  = 0.0
    n_batches   = 0

    for b_start in 1:bs:N
        b_end     = min(b_start + bs - 1, N)
        b_end - b_start + 1 < bs && break      # skip incomplete last batch

        batch_idx = indices[b_start:b_end]

        zero_w_grad!(model.pool)
        zero_a_grad!(model.pool)

        copyto!(input_node.data,  X[:, :, :, batch_idx])
        copyto!(target_node.data, Y[:, batch_idx])

        preds      = forward!(model.chain, input_node, true)
        batch_loss = primal!(loss_fn, preds, target_node)

        loss_fn.out.grad[1] = 1.0f0
        adjoint!(loss_fn, preds, target_node)
        backward!(model.chain, input_node, true)

        optimize!(model.pool, η)
        total_loss += batch_loss
        n_batches  += 1
    end
    return total_loss / n_batches
end

println("[x] Model przed treningiem:")
test(my_model, X_test, Y_test_raw)

println("\n[x] Trenowanie...")
@time for ep in 1:3
    L   = train!(my_model, X_train, Y_train; η=0.02f0)
    acc = test(my_model, X_test, Y_test_raw)
    println("[+] Epoka $ep | Loss: $(round(L, digits=4)) | Acc: $acc%")
end

In [ ]:
using Flux
using Flux: logitcrossentropy
using Statistics   # for mean()

flux_net = Flux.Chain(
  Flux.Conv((3, 3), 1 => 6, pad=1, bias=false),
  Flux.MaxPool((2, 2)),
  Flux.Conv((3, 3), 6 => 16, pad=1, bias=false),
  Flux.MaxPool((2, 2)),
  Flux.flatten,
  Flux.Dense(784 => 84, Flux.relu),
  Flux.Dropout(0.4),
  Flux.Dense(84 => 10)
) |> f32
opt_state = Flux.setup(Descent(0.01), flux_net)

function test_flux(model, X, Y_raw)
    Flux.testmode!(model)
    preds = model(X)
    pred_classes = Flux.onecold(preds, 0:9)
    acc = round(mean(pred_classes .== Y_raw) * 100, digits=2)
    println("Skuteczność: ", acc, "%")
    return acc
end

function train_flux!(model, opt_state, X, Y; batch_size)
    Flux.trainmode!(model)
    N = size(X, 4)
    data_loader = Flux.DataLoader((X, Y), batchsize=batch_size, shuffle=true)
    total_loss = 0.0

    for (x_batch, y_batch) in data_loader
        val, grads = Flux.withgradient(model) do m
            preds = m(x_batch)
            logitcrossentropy(preds, y_batch)
        end
        Flux.update!(opt_state, model, grads[1])
        total_loss += val * size(x_batch, 4)
    end
    return total_loss / N
end

println("[x] Model przed treningiem:")
test_flux(flux_net, X_test, Y_test_raw)

println("[x] Trenowanie...")
@time for ep in 1:3
    L = train_flux!(flux_net, opt_state, X_train, Y_train, batch_size=BATCH_SIZE)
    acc = test_flux(flux_net, X_test, Y_test_raw)
    println("[+] Epoka $ep | Loss: $(round(L, digits=4)) | Acc: $acc%")
end

println("[x] Finalny model:")
test_flux(flux_net, X_test, Y_test_raw)

In [ ]:
using Plots

# https://github.com/zalandoresearch/fashion-mnist
const LABELS = Dict(
    0 => "T-shirt/top",
    1 => "Trouser",
    2 => "Pullover",
    3 => "Dress",
    4 => "Coat",
    5 => "Sandal",
    6 => "Shirt",
    7 => "Sneaker",
    8 => "Bag",
    9 => "Ankle boot"
)

num_images  = 5
indices     = rand(1:size(X_test, 4), num_images)
final_layer = my_model.chain.layers[end]
input_node  = my_model.input       # use model's built-in buffer — no separate declaration

Flux.testmode!(flux_net)

plot_array = []

for idx in indices
    true_label_idx = Y_test_raw[idx]

    # ── My model inference (single sample in a batch-wide buffer) ──
    fill!(input_node.data, 0f0)
    copyto!(@view(input_node.data[:, :, :, 1:1]), @view(X_test[:, :, :, idx:idx]))
    forward!(my_model.chain, input_node, false)
    my_pred_idx = argmax(@view(final_layer.out.data[:, 1])) - 1

    # ── Flux inference ──
    flux_out      = flux_net(X_test[:, :, :, idx:idx])
    flux_pred_idx = Flux.onecold(flux_out, 0:9)[1]

    println("Obrazek $idx:")
    println("  Real:    ", LABELS[true_label_idx])
    println("  My:      ", LABELS[my_pred_idx])
    println("  Flux:    ", LABELS[flux_pred_idx])
    println("=======================")

    img_data   = X_test[:, :, 1, idx]
    title_text = "Real: $(LABELS[true_label_idx])\nMy: $(LABELS[my_pred_idx])\nFlux: $(LABELS[flux_pred_idx])"

    p = heatmap(
        img_data',
        yflip        = true,
        color        = :grays,
        legend       = false,
        aspect_ratio = :equal,
        axis         = false,
        title        = title_text,
        titlefontsize = 8
    )
    push!(plot_array, p)
end

plot(plot_array..., layout=(1, num_images), size=(1200, 300), margin=5Plots.mm)